In [0]:
%pip install -U databricks-sdk -q

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "bank_risk_dev")
dbutils.widgets.text("space_id", "01f1e1ef55ff1a89a53ca836afda12da")
dbutils.widgets.text("runs", "1")
catalog = dbutils.widgets.get("catalog")
space_id = dbutils.widgets.get("space_id").strip()
runs = int(dbutils.widgets.get("runs"))

In [0]:
mv = f"{catalog}.gold"
questions = [
    ("¿Cuál fue el IMOR del segmento masivo en abril de 2026?",
     f"SELECT MEASURE(imor) FROM {mv}.mv_credit_risk WHERE segmento = 'masivo' AND mes = DATE'2026-04-01'"),
    ("¿Cuál es la cartera vencida total?",
     f"SELECT MEASURE(cartera_vencida) FROM {mv}.mv_credit_risk WHERE mes = DATE'2026-10-01'"),
    ("¿Cuál es la morosidad de tarjetas de crédito en el último mes?",
     f"SELECT MEASURE(imor) FROM {mv}.mv_credit_risk WHERE producto = 'tarjeta' AND mes = DATE'2026-10-01'"),
    ("¿En qué mes fue más alta la tasa de fraude?",
     f"SELECT mes, MEASURE(tasa_fraude) AS t FROM {mv}.mv_fraud GROUP BY mes ORDER BY t DESC LIMIT 1"),
    ("¿Cuántos fraudes confirmados hubo en marzo de 2026 por canal?",
     f"SELECT canal, MEASURE(fraudes_confirmados) AS f FROM {mv}.mv_fraud WHERE mes = DATE'2026-03-01' GROUP BY canal"),
    ("¿A qué banco destino se fue más dinero por fraude?",
     f"SELECT banco_destino, MEASURE(monto_fraude) AS m FROM {mv}.mv_fraud GROUP BY banco_destino ORDER BY m DESC LIMIT 1"),
    ("¿Qué regla genera más falsos positivos en proporción?",
     f"SELECT regla, MEASURE(tasa_falsos_positivos) AS t FROM {mv}.mv_alerts GROUP BY regla ORDER BY t DESC LIMIT 1"),
    ("¿Cuántos reportes de operaciones inusuales se generaron en 2026?",
     f"SELECT MEASURE(reportes_pld) FROM {mv}.mv_alerts WHERE mes BETWEEN DATE'2026-01-01' AND DATE'2026-12-01'"),
    ("¿Qué analista tarda más en atender alertas?",
     f"SELECT analista, MEASURE(horas_mediana_atencion) AS h FROM {mv}.mv_alerts GROUP BY analista ORDER BY h DESC LIMIT 1"),
    ("¿Cuál fue el IMOR por calificación de riesgo en octubre de 2026?",
     f"SELECT calificacion, MEASURE(imor) FROM {mv}.mv_credit_risk WHERE mes = DATE'2026-10-01' GROUP BY calificacion"),
]
expected = [[tuple(r) for r in spark.sql(sql).collect()] for _, sql in questions]
print(len(expected), "preguntas listas")

In [0]:
import re
import time
import uuid
from datetime import datetime
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()


def to_num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def same(e, g):
    a, b = to_num(e), to_num(g)
    if a is not None and b is not None:
        return abs(a - b) <= abs(a) * 1e-3 + 1e-9 or abs(a * 100 - b) <= 0.0051
    es, gs = str(e).strip(), str(g).strip()
    if re.match(r"\d{4}-\d{2}-\d{2}", es):
        return gs[:10] == es[:10]
    return es.lower() == gs.lower()


def row_ok(exp_row, got_row):
    pool = list(got_row)
    for e in exp_row:
        i = next((k for k, g in enumerate(pool) if same(e, g)), None)
        if i is None:
            return False
        pool.pop(i)
    return True


def matches(exp, got):
    if len(exp) != len(got):
        return False
    pool = list(got)
    for er in exp:
        i = next((k for k, gr in enumerate(pool) if row_ok(er, gr)), None)
        if i is None:
            return False
        pool.pop(i)
    return True


def ask(question):
    t0 = time.time()
    msg = w.genie.start_conversation_and_wait(space_id=space_id, content=question)
    sql, rows = None, []
    for att in msg.attachments or []:
        if att.query:
            sql = att.query.query
            res = w.genie.get_message_attachment_query_result(
                space_id=space_id,
                conversation_id=msg.conversation_id,
                message_id=msg.message_id,
                attachment_id=att.attachment_id,
            )
            result = res.statement_response.result if res.statement_response else None
            rows = (result.data_array if result else None) or []
            break
    return sql, rows, round(time.time() - t0, 1)

In [0]:
results = []
for run in range(1, runs + 1):
    run_id = uuid.uuid4().hex[:8]
    for qid, ((question, _), exp) in enumerate(zip(questions, expected), start=1):
        try:
            sql, rows, secs = ask(question)
            ok, err = matches(exp, rows), None
        except Exception as e:
            sql, ok, secs, err = None, False, None, str(e)[:500]
        results.append((run_id, run, datetime.now(), qid, question, sql, ok, secs, err))
        print(f"corrida {run} pregunta {qid}: {'OK' if ok else 'FALLA'} ({secs}s)")
        time.sleep(10)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.eval")
schema = ("run_id STRING, run INT, run_ts TIMESTAMP, question_id INT, question STRING, "
          "genie_sql STRING, correct BOOLEAN, seconds DOUBLE, error STRING")
spark.createDataFrame(results, schema).write.mode("append").saveAsTable(f"{catalog}.eval.genie_runs")

display(spark.sql(f"""
SELECT run_id, min(run_ts) AS inicio, count(*) AS preguntas,
       round(100 * avg(CAST(correct AS INT)), 1) AS precision_pct,
       round(avg(seconds), 1) AS segundos_promedio
FROM {catalog}.eval.genie_runs
GROUP BY run_id
ORDER BY inicio
"""))

In [0]:
display(spark.sql(f"""
SELECT run_id, question_id, question, genie_sql, error
FROM {catalog}.eval.genie_runs
WHERE NOT correct
ORDER BY run_ts
"""))